# AI-Based FinTech Adoption & Policy Recommendation Platform
## SME Digital Payment Adoption — Model Training & Evaluation

**Objective:** Predict whether an SME is likely to adopt digital payments.

**Target:** `Digital_Payment_Adoption`
- `1` = Digital payment adopter
- `0` = Digital payment non-adopter

**Models trained**
1. Logistic Regression — interpretable baseline/final model
2. Random Forest — nonlinear comparison model

**Final selected model:** Logistic Regression with decision threshold **0.36**.


## 1. Methodology

```text
Prepared FAT India dataset
        ↓
Define features + target
        ↓
80/20 stratified train/test split
        ↓
Preprocessing
  ├─ Numerical: median imputation + StandardScaler
  └─ Categorical: most-frequent imputation + OneHotEncoder
        ↓
Train Logistic Regression + Random Forest
        ↓
5-fold stratified cross-validation
        ↓
Threshold tuning using training-set OOF predictions
        ↓
Frozen-threshold test evaluation
        ↓
Compare models
        ↓
Select final model
```

The test set is not used for threshold selection.


In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_validate, cross_val_predict
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report
)

RANDOM_STATE = 42
DATA_PATH = r"E:\Pep_Projectdetails\selected_dataset\fintech_ml_final.csv"
MODEL_DIR = r"E:\Pep_Projectdetails\selected_dataset"

print("Data path exists:", os.path.exists(DATA_PATH))


In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nTarget distribution:")
print(df["Digital_Payment_Adoption"].value_counts())


## 2. Features and target

**Numerical:** `s7`, `a6`, `d3c`

**Categorical:** `a5a`, `b5a`, `b5f`, `b5g`, `b5h`, `b13b4`, `b13b5`, `d3a`, `d7a`, `d7b`


In [ ]:
target = "Digital_Payment_Adoption"

numerical_features = ["s7", "a6", "d3c"]

categorical_features = [
    "a5a", "b5a", "b5f", "b5g", "b5h",
    "b13b4", "b13b5", "d3a", "d7a", "d7b"
]

X = df[numerical_features + categorical_features].copy()
y = df[target].astype(int).copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print(y.value_counts())


## 3. Train/test split

An 80/20 stratified split keeps the class proportions approximately stable.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("\ny_train:")
print(y_train.value_counts())
print("\ny_test:")
print(y_test.value_counts())


## 4. Preprocessing

Numerical features use median imputation and standardization.

Categorical features use most-frequent imputation and one-hot encoding.

The preprocessing is inside the model pipeline to reduce data leakage risk.


In [ ]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessor created.")


## 5. Model 1 — Logistic Regression

Chosen because this is a binary classification problem and the model is relatively interpretable. `class_weight="balanced"` helps address the minority non-adopter class.


In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")


## 6. Model 2 — Random Forest

Chosen as a nonlinear comparison model. It can capture interactions and nonlinear patterns that Logistic Regression may miss.


In [ ]:
random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

random_forest_model.fit(X_train, y_train)

print("Random Forest trained successfully.")


## 7. Evaluation function

In [ ]:
def evaluate_model(model, X_eval, y_eval, threshold=0.50):
    prob = model.predict_proba(X_eval)[:, 1]
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y_eval, pred)

    return {
        "Accuracy": accuracy_score(y_eval, pred),
        "Precision": precision_score(y_eval, pred, zero_division=0),
        "Recall": recall_score(y_eval, pred, zero_division=0),
        "F1": f1_score(y_eval, pred, zero_division=0),
        "Macro-F1": f1_score(y_eval, pred, average="macro", zero_division=0),
        "ROC-AUC": roc_auc_score(y_eval, prob),
        "PR-AUC": average_precision_score(y_eval, prob),
        "Class 0 Precision": precision_score(y_eval, pred, pos_label=0, zero_division=0),
        "Class 0 Recall": recall_score(y_eval, pred, pos_label=0, zero_division=0),
        "Confusion Matrix": cm
    }


## 8. Baseline test evaluation — threshold 0.50

In [ ]:
lr_baseline = evaluate_model(logistic_model, X_test, y_test, 0.50)
rf_baseline = evaluate_model(random_forest_model, X_test, y_test, 0.50)

print("LOGISTIC REGRESSION")
print(lr_baseline)

print("\nRANDOM FOREST")
print(rf_baseline)


## 9. 5-fold stratified cross-validation

Cross-validation is performed on the training set. The held-out test set remains untouched.


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "macro_f1": "f1_macro",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

lr_cv = cross_validate(
    logistic_model, X_train, y_train,
    cv=cv, scoring=scoring, n_jobs=-1
)

rf_cv = cross_validate(
    random_forest_model, X_train, y_train,
    cv=cv, scoring=scoring, n_jobs=-1
)

cv_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Macro-F1 mean": [lr_cv["test_macro_f1"].mean(), rf_cv["test_macro_f1"].mean()],
    "Macro-F1 std": [lr_cv["test_macro_f1"].std(), rf_cv["test_macro_f1"].std()],
    "ROC-AUC mean": [lr_cv["test_roc_auc"].mean(), rf_cv["test_roc_auc"].mean()],
    "ROC-AUC std": [lr_cv["test_roc_auc"].std(), rf_cv["test_roc_auc"].std()],
    "PR-AUC mean": [lr_cv["test_pr_auc"].mean(), rf_cv["test_pr_auc"].mean()],
    "PR-AUC std": [lr_cv["test_pr_auc"].std(), rf_cv["test_pr_auc"].std()]
})

display(cv_results)


## 10. Threshold tuning — training data only

Because non-adopters are the minority class, the default 0.50 threshold is not necessarily optimal.

Out-of-fold probabilities from the training set are used to select a threshold by Macro-F1. The selected threshold is then frozen before test evaluation.

Expected thresholds from the completed experiment:
- Logistic Regression: **0.36**
- Random Forest: **0.44**


In [ ]:
lr_oof_prob = cross_val_predict(
    logistic_model, X_train, y_train,
    cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

rf_oof_prob = cross_val_predict(
    random_forest_model, X_train, y_train,
    cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

thresholds = np.arange(0.20, 0.81, 0.01)

def find_best_threshold(y_true, probabilities):
    rows = []
    for threshold in thresholds:
        pred = (probabilities >= threshold).astype(int)
        rows.append({
            "threshold": round(float(threshold), 2),
            "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
            "class0_precision": precision_score(y_true, pred, pos_label=0, zero_division=0),
            "class0_recall": recall_score(y_true, pred, pos_label=0, zero_division=0)
        })

    result = pd.DataFrame(rows)
    best = result.loc[result["macro_f1"].idxmax()]
    return result, best

lr_threshold_results, lr_best = find_best_threshold(y_train, lr_oof_prob)
rf_threshold_results, rf_best = find_best_threshold(y_train, rf_oof_prob)

print("Best Logistic Regression threshold:")
print(lr_best)

print("\nBest Random Forest threshold:")
print(rf_best)


## 11. Final frozen-threshold test evaluation

In [ ]:
LR_THRESHOLD = 0.36
RF_THRESHOLD = 0.44

lr_final = evaluate_model(logistic_model, X_test, y_test, LR_THRESHOLD)
rf_final = evaluate_model(random_forest_model, X_test, y_test, RF_THRESHOLD)

final_results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Threshold": LR_THRESHOLD,
        "Accuracy": lr_final["Accuracy"],
        "Precision": lr_final["Precision"],
        "Recall": lr_final["Recall"],
        "F1": lr_final["F1"],
        "Macro-F1": lr_final["Macro-F1"],
        "ROC-AUC": lr_final["ROC-AUC"],
        "PR-AUC": lr_final["PR-AUC"],
        "Class 0 Precision": lr_final["Class 0 Precision"],
        "Class 0 Recall": lr_final["Class 0 Recall"]
    },
    {
        "Model": "Random Forest",
        "Threshold": RF_THRESHOLD,
        "Accuracy": rf_final["Accuracy"],
        "Precision": rf_final["Precision"],
        "Recall": rf_final["Recall"],
        "F1": rf_final["F1"],
        "Macro-F1": rf_final["Macro-F1"],
        "ROC-AUC": rf_final["ROC-AUC"],
        "PR-AUC": rf_final["PR-AUC"],
        "Class 0 Precision": rf_final["Class 0 Precision"],
        "Class 0 Recall": rf_final["Class 0 Recall"]
    }
])

display(final_results.round(4))

print("\nLogistic Regression confusion matrix:")
print(lr_final["Confusion Matrix"])

print("\nRandom Forest confusion matrix:")
print(rf_final["Confusion Matrix"])


## 12. Final model selection

**Selected model: Logistic Regression**

Random Forest has higher overall accuracy in the completed experiment. Logistic Regression was selected because the project needs an interpretable binary model and places importance on identifying the minority non-adopter group for policy/support analysis.

**Final threshold: 0.36**


In [ ]:
selected_model = logistic_model
selected_threshold = LR_THRESHOLD

final_model_path = os.path.join(
    MODEL_DIR, "final_logistic_regression_model.joblib"
)

threshold_path = os.path.join(
    MODEL_DIR, "final_model_threshold.json"
)

results_path = os.path.join(
    MODEL_DIR, "final_model_results.csv"
)

joblib.dump(selected_model, final_model_path)

with open(threshold_path, "w", encoding="utf-8") as f:
    json.dump({
        "model": "Logistic Regression",
        "target": "Digital_Payment_Adoption",
        "threshold": selected_threshold
    }, f, indent=4)

final_results.to_csv(results_path, index=False)

print("Final model saved:", final_model_path)
print("Threshold saved:", threshold_path)
print("Results saved:", results_path)


## 13. Final Logistic Regression confusion matrix

In [ ]:
cm = lr_final["Confusion Matrix"]

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm)

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Predicted 0", "Predicted 1"])
ax.set_yticklabels(["Actual 0", "Actual 1"])
ax.set_xlabel("Prediction")
ax.set_ylabel("Actual")
ax.set_title("Logistic Regression — Test Confusion Matrix")

for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

print(classification_report(
    y_test,
    (logistic_model.predict_proba(X_test)[:, 1] >= LR_THRESHOLD).astype(int),
    target_names=["Non-Adopter", "Adopter"],
    zero_division=0
))


## 14. Important interpretation

The target is imbalanced: non-adopters are much fewer than adopters. Therefore accuracy alone is not enough.

Important metrics for this project:
- Macro-F1
- Class 0 precision/recall
- ROC-AUC
- PR-AUC
- Confusion matrix

This model is a **prediction/association tool**, not a causal model. A predicted non-adopter does not prove that a particular policy intervention will cause adoption.


## 15. Live/new-SME prediction pattern

For deployment, provide the same 13 input features for a new SME. The saved pipeline performs preprocessing automatically.

```python
probability = selected_model.predict_proba(new_sme)[:, 1][0]
prediction = int(probability >= selected_threshold)

print("Adoption probability:", probability)
print("Prediction:", "Adopter" if prediction == 1 else "Non-Adopter")
```

Do not use predictions on the training dataset as evidence of generalization.


## 16. Reproducibility checklist

- Dataset and target documented
- 13 model features documented
- 80/20 stratified split
- Random seed = 42
- Preprocessing inside pipelines
- Logistic Regression trained
- Random Forest trained
- 5-fold stratified cross-validation
- Threshold tuning performed using training-set OOF predictions
- Final held-out test evaluation
- Final model saved
- Threshold saved
- Results saved

**Repository note:** Do not commit restricted/raw survey microdata to a public GitHub repository unless its terms explicitly permit redistribution. A public repository can contain the training notebook, methodology, results, model artifact where appropriate, and instructions for obtaining the data.
